In [ ]:
!pip install requests

In [ ]:
#문항1
import requests
import csv
from bs4 import BeautifulSoup

hollys_data = []
URL = "https://www.hollys.co.kr/store/korea/korStore2.do"

In [11]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

hollys_data = []
url = 'https://www.hollys.co.kr/store/korea/korStore2.do'

In [ ]:
for page in range(1, 11):
  payload = {'pageNo': page, 'sido': '', 'gugun': '', 'store': ''}

  headers = {
      'User-Agent': (
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,'
          ' like Gecko) Chrome/120.0.0.0 Safari/537.36'
      )
  }

  response = requests.post(url, data=payload, headers=headers)

  if response.status_code != 200:
    print(f'{page}페이지 요청 실패')
    continue

  soup = BeautifulSoup(response.text, 'html.parser')

  store_rows = soup.select('table.tb_store tbody tr')

  for row in store_rows:
    cols = row.find_all('td')
    if len(cols) < 6:
      continue

    store_dict = {
        '지역': cols[0].text.strip(),
        '매장명': cols[1].text.strip(),
        '현황': cols[2].text.strip(),
        '주소': cols[3].text.strip(),
        '매장 서비스': [
            img.get('alt', '').strip()
            for img in cols[4].find_all('img')
            if img.get('alt')
        ],
        '전화번호': cols[5].text.strip(),
    }
    hollys_data.append(store_dict)

  print(f'{page}페이지 수집 완료 (현재 누적: {len(hollys_data)}개)')
  time.sleep(0.5)  

1페이지 수집 완료 (현재 누적: 10개)
2페이지 수집 완료 (현재 누적: 20개)
3페이지 수집 완료 (현재 누적: 30개)
4페이지 수집 완료 (현재 누적: 40개)
5페이지 수집 완료 (현재 누적: 50개)
6페이지 수집 완료 (현재 누적: 60개)
7페이지 수집 완료 (현재 누적: 70개)
8페이지 수집 완료 (현재 누적: 80개)
9페이지 수집 완료 (현재 누적: 90개)
10페이지 수집 완료 (현재 누적: 100개)


In [13]:
df = pd.DataFrame(hollys_data)
df.to_csv('hollys.csv', index=False, encoding='utf-8-sig')
print(f'총 {len(hollys_data)}개 수집 및 hollys.csv 저장 완료!')

총 100개 수집 및 hollys.csv 저장 완료!


In [ ]:
#문항2

In [15]:
URL = 'https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1'
HEADERS = {'User-Agent': 'Mozilla/5.0'}
DELAY = 0.7


def pick(node, selector):
  """특정 요소가 없는 경우 빈 값 처리"""
  el = node.select_one(selector)
  return el.text.strip() if el else ''


def parse(html):
  soup = BeautifulSoup(html, 'html.parser')
  rows = []
  for box in soup.select('div.ss_book_box'):
    img = box.select_one('img')
    rows.append({
        '카테고리': pick(box, 'span.tit_category'),
        '제목': pick(box, 'a.bo3'),
        '저자': pick(box, 'li:nth-of-type(3)'),
        '정가': pick(box, 'li:nth-of-type(4) > span'),
        '할인가격': pick(box, 'span.ss_p2'),
        '이미지': img.get('src', '') if img else '',
    })
  return rows


result = []
for page in range(1, 11):
  r = requests.get(URL, params={'page': page}, headers=HEADERS, timeout=10)
  r.raise_for_status()
  rows = parse(r.text)

  if not rows:
    print(f'{page}페이지가 비어 있어 중단합니다.')
    break

  result.extend(rows)
  print(f'{page}페이지 · 누적 {len(result)}건 수집 완료')
  time.sleep(DELAY)



1페이지 · 누적 50건 수집 완료
2페이지 · 누적 100건 수집 완료
3페이지 · 누적 150건 수집 완료
4페이지 · 누적 200건 수집 완료
5페이지 · 누적 250건 수집 완료
6페이지 · 누적 300건 수집 완료
7페이지 · 누적 350건 수집 완료
8페이지 · 누적 400건 수집 완료
9페이지 · 누적 450건 수집 완료
10페이지 · 누적 500건 수집 완료


In [16]:
pd.DataFrame(result).to_csv(
    'aladin_bestseller.csv', index=False, encoding='utf-8-sig'
)
print('aladin_bestseller.csv 저장 완료!')

print(result[:2]) 

aladin_bestseller.csv 저장 완료!
[{'카테고리': '[국내도서]', '제목': '세네카, 오늘을 빼앗기고 있는 당신에게', '저자': '루키우스 안나이우스 세네카 (지은이), 하와이 대저택 (편역) | 논픽션 | 2026년 7월', '정가': '18,000', '할인가격': '16,200원', '이미지': 'https://image.aladin.co.kr/product/39640/49/cover200/k872130175_1.jpg'}, {'카테고리': '[국내도서]', '제목': '그랬다고 적었다', '저자': '김애란 (지은이) | 문학동네 | 2026년 8월', '정가': '17,000', '할인가격': '15,300원', '이미지': 'https://image.aladin.co.kr/product/40019/36/cover200/k742130236_1.jpg'}]
